# Anti-DreamBooth Full Test (A100 40GB)

Test Anti-DreamBooth with your own images from Google Drive:

1. **Setup** - Install deps, clone repo, download SD model
2. **Load images** - From `My Drive/test_images/`
3. **Baseline** - Train DreamBooth on clean images (attacker scenario)
4. **Attack** - Run ASPL adversarial perturbation
5. **Defense** - Train DreamBooth on perturbed images
6. **Compare** - Side-by-side: baseline vs defended

> **Requirements:** A100 40GB, HF token in Colab secrets, 4-10 face images in `My Drive/test_images/`

---
## 0. GPU Check

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory
    print(f"Memory: {total_mem / 1e9:.1f} GB")
    assert total_mem > 35e9, "This notebook requires A100 40GB!"
    print("\n>>> A100 40GB confirmed.")
else:
    raise RuntimeError("No GPU! Runtime > Change runtime type > A100")

---
## 1. Install Dependencies & Clone Repo

In [ ]:
%%time
!pip uninstall -y diffusers 2>/dev/null
!pip install -q --no-deps diffusers
!pip install -q "transformers>=4.40.0,<5" "accelerate>=0.28.0" safetensors datasets ftfy Jinja2 tqdm tensorboard
import importlib, diffusers; importlib.reload(diffusers)
print(f"diffusers: {diffusers.__version__}\n>>> Dependencies installed.")

In [ ]:
import torch, diffusers, transformers, accelerate
print(f"torch: {torch.__version__}, diffusers: {diffusers.__version__}, transformers: {transformers.__version__}, accelerate: {accelerate.__version__}")
print(f"CUDA: {torch.version.cuda}, SDPA: {hasattr(torch.nn.functional, 'scaled_dot_product_attention')}")
from diffusers import AutoencoderKL, DDPMScheduler, DiffusionPipeline, UNet2DConditionModel
from diffusers.optimization import get_scheduler
from diffusers.utils.import_utils import is_xformers_available
print(">>> All imports OK.")

In [ ]:
import os
REPO_URL = "https://github.com/superbabiix/anti-dreambooth-fork.git"
REPO_DIR = "/content/Anti-DreamBooth-Fork"
BRANCH = "claude/project-review-research-pavb5"
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
    print(f"Updated to latest origin/{BRANCH}")
os.chdir(REPO_DIR)
!git log --oneline -3

---
## 2. Download Stable Diffusion Model

In [ ]:
%%time
import torch, os
SD_LOCAL_PATH = "./stable-diffusion/sd-model"
if not os.path.exists(SD_LOCAL_PATH):
    from diffusers import DiffusionPipeline
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
        print("Authenticated via Colab secret.\n")
    except Exception:
        print("HF_TOKEN not found. Trying without auth...\n")
    pipe = None
    for mid in ["stabilityai/stable-diffusion-2-1-base", "Manojb/stable-diffusion-2-1-base", "stable-diffusion-v1-5/stable-diffusion-v1-5"]:
        try:
            print(f"Trying {mid} ...")
            pipe = DiffusionPipeline.from_pretrained(mid, torch_dtype=torch.float16)
            print(f">>> Loaded: {mid}"); break
        except Exception as e:
            print(f"  Failed: {e.__class__.__name__}\n")
    if pipe is None: raise RuntimeError("All model sources failed!")
    os.makedirs(SD_LOCAL_PATH, exist_ok=True); pipe.save_pretrained(SD_LOCAL_PATH); del pipe; torch.cuda.empty_cache()
else:
    print(f"Model exists at {SD_LOCAL_PATH}")
!ls {SD_LOCAL_PATH}

---
## 3. Load Your Images from Google Drive

Put 4-10 face images in `My Drive/test_images/`. Auto-split and resized to 512x512.

In [ ]:
import shutil, numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt

DRIVE_FOLDER = "test_images"
RESOLUTION = 512
MAX_IMAGES = 10
SUBJECT_TAG = "custom"

from google.colab import drive
drive.mount("/content/drive")

image_folder = Path(f"/content/drive/My Drive/{DRIVE_FOLDER}")
if not image_folder.exists():
    raise FileNotFoundError(f"Create My Drive/{DRIVE_FOLDER}/ and add face images.")

image_files = sorted(list(image_folder.glob("*.jpg")) + list(image_folder.glob("*.png")) + list(image_folder.glob("*.jpeg")))
if not image_files:
    raise FileNotFoundError(f"No images in {DRIVE_FOLDER}/ (.jpg/.png/.jpeg)")
if len(image_files) > MAX_IMAGES:
    print(f"Found {len(image_files)} images, using first {MAX_IMAGES}")
    image_files = image_files[:MAX_IMAGES]
print(f"Using {len(image_files)} images\n")

print("--- Resolution Check ---")
for p in image_files:
    w, h = Image.open(p).size
    print(f"  {p.name}: {w}x{h} {'[OK]' if (w,h)==(RESOLUTION,RESOLUTION) else f'[resize {w}x{h}]'}")

_c = np.array(Image.open(image_files[0]))
print(f"\nSample: min={_c.min()}, max={_c.max()}, mean={_c.mean():.1f}, shape={_c.shape}")
if _c.max() < 10: raise ValueError("Images appear black!")

mid = len(image_files) // 2
if mid < 2: raise ValueError(f"Need >= 4 images (got {len(image_files)})")

for subset, files in [("set_A", image_files[:mid]), ("set_B", image_files[mid:])]:
    dest = Path(f"data/custom/{subset}")
    if dest.exists(): shutil.rmtree(dest)
    dest.mkdir(parents=True)
    for f in files:
        Image.open(f).convert("RGB").resize((RESOLUTION, RESOLUTION)).save(dest / f"{f.stem}.png")

TRAIN_DIR = "data/custom/set_A"
ADV_DIR = "data/custom/set_B"
print(f"\nSet A ({mid} imgs): {[f.name for f in image_files[:mid]]}")
print(f"Set B ({len(image_files)-mid} imgs): {[f.name for f in image_files[mid:]]}")

def show_images(directory, title, max_images=5):
    d = Path(directory)
    if not d.exists(): print(f"[SKIP] {directory}"); return
    imgs = sorted(list(d.glob("*.png")) + list(d.glob("*.jpg")))[:max_images]
    if not imgs: print(f"[SKIP] No images in {directory}"); return
    fig, axes = plt.subplots(1, len(imgs), figsize=(4*len(imgs), 4))
    if len(imgs) == 1: axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for ax, p in zip(axes, imgs): ax.imshow(Image.open(p)); ax.set_title(p.name, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()

show_images(TRAIN_DIR, "Set A (Training Reference)")
show_images(ADV_DIR, "Set B (Will Be Perturbed)")
print("\n>>> Images ready.")

---
## 4. Baseline: Train DreamBooth on Clean Images

In [ ]:
%%time
import shutil
from pathlib import Path
clean_all = f"data/{SUBJECT_TAG}_clean_all"
os.makedirs(clean_all, exist_ok=True)
for f in Path(TRAIN_DIR).glob("*"): shutil.copy2(f, clean_all)
for f in Path(ADV_DIR).glob("*"): shutil.copy2(f, clean_all)
print(f"Combined {len(list(Path(clean_all).iterdir()))} clean images")

!accelerate launch train_dreambooth.py \
  --pretrained_model_name_or_path="./stable-diffusion/sd-model" \
  --train_text_encoder \
  --instance_data_dir="{clean_all}" \
  --class_data_dir="data/class-person" \
  --output_dir="outputs/BASELINE/{SUBJECT_TAG}_DREAMBOOTH" \
  --with_prior_preservation --prior_loss_weight=1.0 \
  --instance_prompt="a photo of sks person" \
  --class_prompt="a photo of person" \
  --inference_prompt="a photo of sks person;a dslr portrait of sks person" \
  --resolution=512 --train_batch_size=2 --gradient_accumulation_steps=1 \
  --learning_rate=5e-7 --lr_scheduler="constant" --lr_warmup_steps=0 \
  --num_class_images=200 --max_train_steps=1000 --checkpointing_steps=500 \
  --center_crop --mixed_precision=bf16 --prior_generation_precision=bf16 --sample_batch_size=8

print("\n>>> Baseline training complete.")

---
## 5. ASPL Attack: Generate Adversarial Perturbations

In [ ]:
%%time
OUTPUT_DIR = f"outputs/ASPL/{SUBJECT_TAG}_ADVERSARIAL"
!mkdir -p {OUTPUT_DIR}
!cp -r {TRAIN_DIR} {OUTPUT_DIR}/image_clean
!cp -r {ADV_DIR} {OUTPUT_DIR}/image_before_adding_noise

!accelerate launch attacks/aspl.py \
  --pretrained_model_name_or_path="./stable-diffusion/sd-model" \
  --instance_data_dir_for_train="{TRAIN_DIR}" \
  --instance_data_dir_for_adversarial="{ADV_DIR}" \
  --instance_prompt="a photo of sks person" \
  --class_data_dir="data/class-person" --num_class_images=200 \
  --class_prompt="a photo of person" \
  --output_dir="{OUTPUT_DIR}" \
  --center_crop --with_prior_preservation --prior_loss_weight=1.0 \
  --resolution=512 --train_text_encoder --train_batch_size=1 \
  --max_train_steps=50 --max_f_train_steps=3 --max_adv_train_steps=6 \
  --checkpointing_iterations=10 --learning_rate=5e-7 --pgd_alpha=5e-3 --pgd_eps=5e-2

print("\n>>> ASPL attack complete.")

In [ ]:
print("Perturbed checkpoints:")
!ls outputs/ASPL/{SUBJECT_TAG}_ADVERSARIAL/noise-ckpt/
show_images(ADV_DIR, "Original Set B (Clean)")
show_images(f"outputs/ASPL/{SUBJECT_TAG}_ADVERSARIAL/noise-ckpt/50", "Set B After ASPL (Step 50)")

---
## 6. Defense: Train DreamBooth on Perturbed Images

In [ ]:
%%time
!accelerate launch train_dreambooth.py \
  --pretrained_model_name_or_path="./stable-diffusion/sd-model" \
  --train_text_encoder \
  --instance_data_dir="outputs/ASPL/{SUBJECT_TAG}_ADVERSARIAL/noise-ckpt/50" \
  --class_data_dir="data/class-person" \
  --output_dir="outputs/ASPL/{SUBJECT_TAG}_DREAMBOOTH" \
  --with_prior_preservation --prior_loss_weight=1.0 \
  --instance_prompt="a photo of sks person" \
  --class_prompt="a photo of person" \
  --inference_prompt="a photo of sks person;a dslr portrait of sks person" \
  --resolution=512 --train_batch_size=2 --gradient_accumulation_steps=1 \
  --learning_rate=5e-7 --lr_scheduler="constant" --lr_warmup_steps=0 \
  --num_class_images=200 --max_train_steps=1000 --checkpointing_steps=500 \
  --center_crop --mixed_precision=bf16 --prior_generation_precision=bf16 --sample_batch_size=8

print("\n>>> Defense training complete.")

---
## 7. Inference

In [ ]:
%%time
import glob
for label, pattern, outdir in [
    ("Baseline", f"outputs/BASELINE/{SUBJECT_TAG}_DREAMBOOTH/checkpoint-*", "outputs/infer_baseline"),
    ("Defended", f"outputs/ASPL/{SUBJECT_TAG}_DREAMBOOTH/checkpoint-*", "outputs/infer_defended"),
]:
    ckpts = sorted(glob.glob(pattern))
    if ckpts:
        ckpt = ckpts[-1]
        print(f"{label}: {ckpt}")
        !python infer.py --model_path="{ckpt}" --output_dir="{outdir}"
        print(f">>> {label} inference complete.\n")
    else:
        print(f"{label}: No checkpoint found!\n")

---
## 8. Visual Comparison: Baseline vs Defended

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

def get_prompt_dirs(d):
    p = Path(d)
    return sorted([x for x in p.iterdir() if x.is_dir()]) if p.exists() else []

bp = get_prompt_dirs("outputs/infer_baseline")
dp = get_prompt_dirs("outputs/infer_defended")
n = min(len(bp), len(dp))
if n == 0:
    print("No inference results. Run inference cell first.")
else:
    for i in range(n):
        bi = sorted(bp[i].glob("*.png"))[:4]
        di = sorted(dp[i].glob("*.png"))[:4]
        if not bi or not di: continue
        cols = max(len(bi), len(di))
        fig, axes = plt.subplots(2, cols, figsize=(4*cols, 8))
        fig.suptitle(bp[i].name.replace('_', ' '), fontsize=13, fontweight='bold')
        for j in range(cols):
            if j < len(bi): axes[0][j].imshow(Image.open(bi[j]))
            axes[0][j].axis('off')
            if j == 0: axes[0][j].set_ylabel("Baseline", fontsize=12, fontweight='bold')
            if j < len(di): axes[1][j].imshow(Image.open(di[j]))
            axes[1][j].axis('off')
            if j == 0: axes[1][j].set_ylabel("Defended", fontsize=12, fontweight='bold')
        plt.tight_layout(); plt.show()
    print(">>> Defended row should show degraded faces.")

---
## 9. Perturbation Analysis

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt

clean_dir = Path(ADV_DIR)
noisy_dir = Path(f"outputs/ASPL/{SUBJECT_TAG}_ADVERSARIAL/noise-ckpt/50")
cf = sorted(list(clean_dir.glob("*.png")) + list(clean_dir.glob("*.jpg")))
nf = sorted(noisy_dir.glob("*.png")) if noisy_dir.exists() else []
if cf and nf:
    n = min(len(cf), len(nf))
    fig, axes = plt.subplots(3, n, figsize=(4*n, 12))
    if n == 1: axes = [[ax] for ax in axes]
    fig.suptitle("Perturbation Analysis", fontsize=14, fontweight='bold')
    for i in range(n):
        ci = np.array(Image.open(cf[i]).resize((512,512))).astype(float)
        ni = np.array(Image.open(nf[i]).resize((512,512))).astype(float)
        d = np.abs(ni - ci)
        axes[0][i].imshow(ci.astype(np.uint8)); axes[0][i].set_title("Clean"); axes[0][i].axis('off')
        axes[1][i].imshow(ni.astype(np.uint8)); axes[1][i].set_title("Perturbed"); axes[1][i].axis('off')
        axes[2][i].imshow(np.clip(d*10,0,255).astype(np.uint8)); axes[2][i].set_title(f"Diff x10 (max={d.max():.1f})"); axes[2][i].axis('off')
        print(f"Image {i}: L_inf={d.max():.2f}, L2={np.sqrt((d**2).mean()):.2f}, Mean={d.mean():.2f}")
    plt.tight_layout(); plt.show()
else:
    print("No clean/noisy pairs found.")

---
## 10. Summary

In [ ]:
import os
from pathlib import Path
def dir_size(p):
    t = 0
    for dp, dn, fns in os.walk(p):
        for f in fns: t += os.path.getsize(os.path.join(dp, f))
    return t / (1024**3)
print("=" * 60)
print(f"ANTI-DREAMBOOTH TEST SUMMARY ({SUBJECT_TAG})")
print("=" * 60)
for name, path in [("SD model", "./stable-diffusion/sd-model"), ("Baseline", f"outputs/BASELINE/{SUBJECT_TAG}_DREAMBOOTH"),
    ("ASPL attack", f"outputs/ASPL/{SUBJECT_TAG}_ADVERSARIAL"), ("Defended", f"outputs/ASPL/{SUBJECT_TAG}_DREAMBOOTH"),
    ("Baseline infer", "outputs/infer_baseline"), ("Defended infer", "outputs/infer_defended")]:
    print(f"  {name:25s} {dir_size(path):.2f} GB" if os.path.exists(path) else f"  {name:25s} [NOT FOUND]")
for label, d in [("Baseline", "outputs/infer_baseline"), ("Defended", "outputs/infer_defended")]:
    if os.path.exists(d): print(f"  {label} images: {sum(1 for _ in Path(d).rglob('*.png'))}")
print(">>> Done!")

---
## 11. Cleanup (Optional)

In [ ]:
# !rm -rf outputs/ stable-diffusion/ data/class-person/ data/custom/
# print("Cleaned up.")